# Practice: Audit a Preparation Pipeline End to End

**Computational Text Analysis Pack · Module 14 — Preparing Text, Properly**

*Technique:* Tracing one document through every preparation step and costing each

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/14e_practice_pipeline_audit.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **A colleague hands you a cleaned corpus and a frequency list. What do you
> need to know about the preparation before you trust the numbers?**

This notebook builds a pipeline, then audits it — measuring what each step
removes, so you can state the cost of each decision instead of assuming it was
free.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import re
import numpy as np
raw = " ".join(df_colonial["text"].head(400).astype(str))
print(f"raw input: {len(raw):,} characters")

STEPS = []
def record(name, tokens, note):
    STEPS.append({"step": name, "tokens": len(tokens),
                  "types": len(set(tokens)), "note": note})
    return tokens

t = record("1. unicode tokenise", re.findall(r"\w+", raw.lower()),
           "baseline: \\w+ keeps diacritics")
t2 = record("2. drop numbers", [w for w in t if not w.isdigit()],
            "loses dates and page numbers")
STOP = {"the","a","an","and","or","of","to","in","on","for","with","is","was",
        "are","were","be","been","it","its","this","that","as","at","by","from"}
t3 = record("3. drop stopwords", [w for w in t2 if w not in STOP],
            "destroys the stylometry signal (module 9)")
t4 = record("4. min length 3", [w for w in t3 if len(w) >= 3],
            "loses short content words too")
t5 = record("5. crude stem", [re.sub(r"(ing|ed|s)$", "", w) for w in t4],
            "merges distinct words; output may not be words")

audit = pd.DataFrame(STEPS)
audit["tokens_kept"] = audit["tokens"] / audit["tokens"].iloc[0]
audit["types_kept"] = audit["types"] / audit["types"].iloc[0]
print(audit.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(audit))
ax.plot(x, audit["tokens_kept"], "-o", label="tokens kept")
ax.plot(x, audit["types_kept"], "-s", label="types kept")
ax.set_xticks(x); ax.set_xticklabels([s.split(". ")[1] for s in audit["step"]],
                                     rotation=25, ha="right", fontsize=8)
ax.set_ylabel("proportion of the original"); ax.legend()
ax.set_title("What each preparation step costs")
plt.tight_layout()
print("Types fall faster than tokens at the stemming step — that is the merging")
print("happening. Whether it helps depends on the task; module 14a measured it.")

In [ ]:
# The audit question that matters most: what did we lose that we needed?
lost_by_stop = Counter(w for w in t2 if w in STOP)
print("Top items removed by the stopword list:")
for w, n in lost_by_stop.most_common(10):
    print(f"   {w:<12}{n:>7,}")
print()
print(f"That is {sum(lost_by_stop.values()):,} tokens, "
      f"{sum(lost_by_stop.values())/len(t2):.1%} of the corpus.")
print()
print("Module 9 uses exactly these words as the authorship signal. A corpus")
print("prepared this way CANNOT be used for stylometry, and nothing about the")
print("frequency list would tell you so.")

In [ ]:
# And the step nobody mentions: what did the tokeniser do to non-English text?
sample = " ".join(df[df["is_nigerian"]]["text"].head(80).astype(str))
uni = re.findall(r"\w+", sample.lower())
ascii_only = re.findall(r"[a-z]+", sample.lower())
print(f"unicode-aware tokens : {len(uni):,}  ({len(set(uni)):,} types)")
print(f"ascii-only tokens    : {len(ascii_only):,}  ({len(set(ascii_only)):,} types)")
print(f"inflation in types from splitting on diacritics: "
      f"{len(set(ascii_only))/max(len(set(uni)),1) - 1:+.1%}")
print()
print("The ascii pipeline does not just lose characters — it INVENTS types by")
print("splitting words at accents, which inflates vocabulary and corrupts every")
print("richness measure downstream.")

## What to ask about someone else's corpus

1. **Which tokeniser, and is it Unicode-aware?** If the corpus contains African
   languages and the answer is `[a-z]+`, the counts are wrong.
2. **Which stopword list?** If any, the corpus cannot support stylometry.
3. **Stemmed or lemmatised, and with what?** Stems are not words and cannot be
   looked up in a concordance.
4. **What was dropped before you saw it?** Numbers, short tokens, front matter.
5. **What is the OCR error rate?** Module 14d showed a clean-looking frequency
   list sitting on top of a badly corrupted mid-frequency range.

**Defensible:** "The corpus was prepared with [pipeline]; this retains X% of
tokens and Y% of types, and precludes stylometric analysis because function
words were removed."

**Not defensible:** "the corpus was cleaned".

### 🔧 Your turn 1

Reorder the pipeline — stem before dropping stopwords — and see whether the final token count changes.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

It usually does, because stemming can turn a content word into something that
matches a stopword ('being' to 'be'), and the order determines whether it is
caught. That order-dependence is the practical argument for recording the
pipeline as an ordered list rather than a set of adjectives: two projects that
both say they "removed stopwords and stemmed" may have processed the same corpus
into measurably different data.

</details>